# Case Study - 3

**Agenda**
- Case Study: Predicting Loan Approval Status
   - Objective
   - Data Description
   - Data Loading and Processing
- Decision Tree Classifier
   - Entropy
   - Gini Index
   - Pruning Techniques
   - Hyperparameter Tuning
- Bagging with Random Forest
   - Bootstrapping and Bagging Mechanism
   - Feature Importance in Random Forest
   - Hyperparameter Tuning
   - Implementing Grid Search CV for Random Forest
-  Boosting
   - AdaBoost
   - Gradient Boosting
   - XGBoost (Extreme Gradient Boosting)
   - Hyperparameter Tuning for Boosting
- Conclusion

## Case Study: Predicting Loan Approval Status

### Objective

* The goal of this case study is to predict whether a loan application will be approved based on various applicant attributes. The data consists of information about the applicant, including income, loan amount, credit history, and more. The target variable is whether the loan application was approved (Loan_Status), which is a binary classification problem.

### Data Description

**The dataset contains the following features:**

- Loan_ID: Unique Loan ID
- Gender: Male/Female
- Married: Applicant married (Yes/No)
- Dependents: Number of dependents
- Education: Applicant Education (Graduate/Not Graduate)
- Self_Employed: Self-employed (Yes/No)
- ApplicantIncome: Applicant income
- CoapplicantIncome: Coapplicant income
- LoanAmount: Loan amount in thousands
- Loan_Amount_Term: Term of loan in months
- Credit_History: Credit history meets guidelines (1/0)
- Property_Area: Urban/Semi-Urban/Rural
- Loan_Status: Loan approved (Y/N)

### Data Loading and Processing

[Download Dataset From Here](https://drive.google.com/file/d/1PA2n3Rdu5U4Vko6btlW1Jqn8R76OwLqs/view?usp=sharing)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv('/content/Loan_Prediction.csv')

In [ ]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [ ]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


### Convert categorical variables to numerical using LabelEncoder

**Why Convert Categorical Values to Numerical Values?**

- Machine Learning Algorithms Require Numerical Input:
   - Decision Trees can inherently handle categorical data, but encoding might still be necessary depending on the implementation.
   - Bagging (e.g., Random Forest) often uses Decision Trees, and categorical variables should be encoded for algorithms like Scikit-learn.
   - Boosting (e.g., Gradient Boosting, AdaBoost) requires categorical data to be numerically encoded because gradient-based algorithms need numerical inputs.

In [ ]:
label_encoder = LabelEncoder()
df['Gender'] = label_encoder.fit_transform(df['Gender'])
df['Married'] = label_encoder.fit_transform(df['Married'])
df['Education'] = label_encoder.fit_transform(df['Education'])
df['Self_Employed'] = label_encoder.fit_transform(df['Self_Employed'])
df['Property_Area'] = label_encoder.fit_transform(df['Property_Area'])
df['Loan_Status'] = label_encoder.fit_transform(df['Loan_Status'])  # Target variable

In [ ]:
# Define the features and target variable
X = df.drop(columns=['Loan_ID', 'Loan_Status'])
y = df['Loan_Status']

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## Decision Tree Classifier

### Entropy and Information Gain

* Entropy is a measure of impurity in a dataset. The information gain is the decrease in entropy after splitting the dataset based on a feature.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Convert 'Dependents' column to numerical values
X_train['Dependents'] = X_train['Dependents'].replace('3+', 3).astype(float)
X_test['Dependents'] = X_test['Dependents'].replace('3+', 3).astype(float)

# Now you can proceed with fitting the model
dt_entropy = DecisionTreeClassifier(criterion='entropy', random_state=42)
dt_entropy.fit(X_train, y_train)

# Predictions and evaluation
y_pred_entropy = dt_entropy.predict(X_test)
accuracy_entropy = accuracy_score(y_test, y_pred_entropy)

print("Decision Tree with Entropy - Accuracy: ", accuracy_entropy)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_entropy))
print("Classification Report:\n", classification_report(y_test, y_pred_entropy))


Decision Tree with Entropy - Accuracy:  0.6702702702702703
Confusion Matrix:
 [[35 30]
 [31 89]]
Classification Report:
               precision    recall  f1-score   support

           0       0.53      0.54      0.53        65
           1       0.75      0.74      0.74       120

    accuracy                           0.67       185
   macro avg       0.64      0.64      0.64       185
weighted avg       0.67      0.67      0.67       185



###  Gini Index

The Gini Index is another impurity measure used to select the best split.

In [ ]:
# Initialize the Decision Tree with Gini criterion
dt_gini = DecisionTreeClassifier(criterion='gini', random_state=42)
dt_gini.fit(X_train, y_train)

# Predictions and evaluation
y_pred_gini = dt_gini.predict(X_test)
accuracy_gini = accuracy_score(y_test, y_pred_gini)

print("Decision Tree with Gini Index - Accuracy: ", accuracy_gini)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gini))
print("Classification Report:\n", classification_report(y_test, y_pred_gini))


Decision Tree with Gini Index - Accuracy:  0.6594594594594595
Confusion Matrix:
 [[34 31]
 [32 88]]
Classification Report:
               precision    recall  f1-score   support

           0       0.52      0.52      0.52        65
           1       0.74      0.73      0.74       120

    accuracy                           0.66       185
   macro avg       0.63      0.63      0.63       185
weighted avg       0.66      0.66      0.66       185



This code initializes a Decision Tree classifier using the Gini index as the splitting criterion and trains it on the X_train and y_train data. After making predictions on the test set (X_test), the accuracy, confusion matrix, and classification report are printed to evaluate the performance of the Decision Tree model using the Gini index.

### Pruning Techniques

* Pruning helps to reduce the size of the decision tree and prevent overfitting.

   * Pre-Pruning: Limit the depth or number of samples required for a split.
   * Post-Pruning: Perform pruning after the tree is fully grown, removing nodes that do not improve the model’s performance.


In [ ]:
# Pre-pruning by limiting max_depth and min_samples_split
dt_pruned = DecisionTreeClassifier(max_depth=4, min_samples_split=10, random_state=42)
dt_pruned.fit(X_train, y_train)

# Predictions and evaluation
y_pred_pruned = dt_pruned.predict(X_test)
accuracy_pruned = accuracy_score(y_test, y_pred_pruned)

print("Decision Tree with Pre-Pruning - Accuracy: ", accuracy_pruned)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_pruned))
print("Classification Report:\n", classification_report(y_test, y_pred_pruned))


Decision Tree with Pre-Pruning - Accuracy:  0.7621621621621621
Confusion Matrix:
 [[ 24  41]
 [  3 117]]
Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.37      0.52        65
           1       0.74      0.97      0.84       120

    accuracy                           0.76       185
   macro avg       0.81      0.67      0.68       185
weighted avg       0.79      0.76      0.73       185



### Hyperparameter Tuning

We will perform Grid Search to find the best hyperparameters for the Decision Tree.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    'max_depth': [3, 5, 7, 9],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

# Perform Grid Search
grid_search_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5)
grid_search_dt.fit(X_train, y_train)

# Best parameters and accuracy
print("Best Parameters: ", grid_search_dt.best_params_)
best_dt = grid_search_dt.best_estimator_
y_pred_best_dt = best_dt.predict(X_test)
accuracy_best_dt = accuracy_score(y_test, y_pred_best_dt)

print("Best Decision Tree - Accuracy: ", accuracy_best_dt)


Best Parameters:  {'criterion': 'entropy', 'max_depth': 3, 'min_samples_split': 2}
Best Decision Tree - Accuracy:  0.7783783783783784


/usr/local/lib/python3.10/dist-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,



This code performs hyperparameter tuning for a Decision Tree classifier using GridSearchCV. It searches for the best combination of max_depth, min_samples_split, and criterion over 5-fold cross-validation. After finding the best parameters, it trains the best model on the training data and predicts on the test set, followed by printing the best parameters and the model's accuracy on the test data.

Overfitting is handled by pruning and limiting tree complexity.

## Bagging with Random Forest

Bagging stands for Bootstrap Aggregating and is a way of reducing variance by training multiple models on different subsets of data.

### Bootstrapping and Bagging Mechanism

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, bootstrap=True)
rf_model.fit(X_train, y_train)

# Predictions and evaluation
y_pred_rf = rf_model.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("Random Forest (Bagging) - Accuracy: ", accuracy_rf)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))


Random Forest (Bagging) - Accuracy:  0.772972972972973
Confusion Matrix:
 [[ 30  35]
 [  7 113]]
Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.46      0.59        65
           1       0.76      0.94      0.84       120

    accuracy                           0.77       185
   macro avg       0.79      0.70      0.72       185
weighted avg       0.78      0.77      0.75       185



This code initializes and trains a Random Forest classifier with 100 estimators using bagging (bootstrap=True). After fitting the model on the training data (X_train and y_train), it predicts on the test data (X_test). The accuracy, confusion matrix, and classification report are then printed to evaluate the Random Forest model's performance.

### Feature Importance in Random Forest

Random Forest provides feature importance, which helps in identifying the most relevant features.

In [ ]:
# Feature Importance
feature_importance = rf_model.feature_importances_
features = X_train.columns

# Display the feature importance
for feature, importance in zip(features, feature_importance):
    print(f"{feature}: {importance}")


Gender: 0.03237461911090116
Married: 0.024655336044844635
Dependents: 0.05792535556293866
Education: 0.0208657630469004
Self_Employed: 0.028964128439403006
ApplicantIncome: 0.18967338911963633
CoapplicantIncome: 0.10260396419526549
LoanAmount: 0.18677866282880018
Loan_Amount_Term: 0.051196804516640604
Credit_History: 0.2555267533187809
Property_Area: 0.049435223815888774


This code calculates and displays the feature importance from a trained Random Forest model (rf_model). It retrieves the importance scores for each feature and prints them alongside their corresponding feature names from X_train, showing how much each feature contributes to the model's predictions.

### Hyperparameter Tuning

In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'min_samples_split': [2, 5, 10]
}

grid_search_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5)
grid_search_rf.fit(X_train, y_train)

# Best parameters
print("Best Parameters (Random Forest): ", grid_search_rf.best_params_)


/usr/local/lib/python3.10/dist-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best Parameters (Random Forest):  {'max_depth': 4, 'min_samples_split': 2, 'n_estimators': 200}


This code performs hyperparameter tuning for a Random Forest classifier using GridSearchCV. It searches for the best combination of n_estimators, max_depth, and min_samples_split values over 5-fold cross-validation. The best parameters found during this search are then printed.

### Implementing Grid Search CV for Random Forest

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define the Random Forest model
rf_model = RandomForestClassifier(random_state=42)

# Set up the parameter grid to search over
param_grid = {
    'n_estimators': [50, 100, 200],  # Number of trees
    'max_depth': [None, 10, 20, 30],  # Depth of the trees
    'min_samples_split': [2, 5, 10],  # Minimum samples required to split a node
    'min_samples_leaf': [1, 2, 4],  # Minimum samples required at each leaf node
    'max_features': ['sqrt', 'log2', None]  # Number of features to consider at each split
}

# Apply GridSearchCV to tune Random Forest hyperparameters
grid_search_rf = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2, scoring='accuracy')

# Fit GridSearchCV on the training data
grid_search_rf.fit(X_train, y_train)

# Get the best parameters and the best model
best_params_rf = grid_search_rf.best_params_
best_rf_model = grid_search_rf.best_estimator_

# Print the best parameters
print(f"Best parameters for Random Forest: {best_params_rf}")

# Use the best model to make predictions on the test set
y_pred_rf_grid = best_rf_model.predict(X_test)

# Evaluate the model's performance
accuracy_rf_grid = accuracy_score(y_test, y_pred_rf_grid)
print("Random Forest Accuracy (with Grid Search CV): ", accuracy_rf_grid)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf_grid))
print("Classification Report:\n", classification_report(y_test, y_pred_rf_grid))


Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Best parameters for Random Forest: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}
Random Forest Accuracy (with Grid Search CV):  0.7837837837837838
Confusion Matrix:
 [[ 27  38]
 [  2 118]]
Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.42      0.57        65
           1       0.76      0.98      0.86       120

    accuracy                           0.78       185
   macro avg       0.84      0.70      0.71       185
weighted avg       0.82      0.78      0.76       185



## Boosting

### AdaBoost

AdaBoost increases the weight of misclassified samples to focus on difficult cases.

In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Assuming X is your original DataFrame with missing values
# If X_train is a separate DataFrame derived from X, replace X with X_train in the following line

# Create an imputer to fill missing values with the mean of each column
imputer = SimpleImputer(strategy='mean')

# Fit the imputer to your data and transform X_train
X_train = imputer.fit_transform(X_train)

# Initialize AdaBoost
ada_model = AdaBoostClassifier(n_estimators=100, random_state=42)

# Now you can fit the model without the ValueError
ada_model.fit(X_train, y_train)

# Apply the imputer to X_test (This is the important step you missed)
X_test = imputer.transform(X_test)

# Predictions and evaluation
y_pred_ada = ada_model.predict(X_test)
accuracy_ada = accuracy_score(y_test, y_pred_ada)

print("AdaBoost Accuracy: ", accuracy_ada)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ada))
print("Classification Report:\n", classification_report(y_test, y_pred_ada))

/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoost Accuracy:  0.7837837837837838
Confusion Matrix:
 [[ 32  33]
 [  7 113]]
Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.49      0.62        65
           1       0.77      0.94      0.85       120

    accuracy                           0.78       185
   macro avg       0.80      0.72      0.73       185
weighted avg       0.79      0.78      0.77       185




This code handles missing values using mean imputation and trains an AdaBoost classifier with 100 estimators. The SimpleImputer fills missing values in X_train and X_test with the mean of each column. After fitting the model on the imputed X_train, predictions are made on the imputed X_test. The accuracy, confusion matrix, and classification report are printed to evaluate the AdaBoost model's performance.

### Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Initialize Gradient Boosting
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)

# Predictions and evaluation
y_pred_gb = gb_model.predict(X_test)
accuracy_gb = accuracy_score(y_test, y_pred_gb)

print("Gradient Boosting Accuracy: ", accuracy_gb)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gb))
print("Classification Report:\n", classification_report(y_test, y_pred_gb))


Gradient Boosting Accuracy:  0.7405405405405405
Confusion Matrix:
 [[ 26  39]
 [  9 111]]
Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.40      0.52        65
           1       0.74      0.93      0.82       120

    accuracy                           0.74       185
   macro avg       0.74      0.66      0.67       185
weighted avg       0.74      0.74      0.72       185



This code initializes and trains a Gradient Boosting classifier with 100 estimators. After fitting the model to the training data (X_train and y_train), it makes predictions on the test set (X_test). The model's accuracy, confusion matrix, and classification report are printed to evaluate its performance.

### XGBoost (Extreme Gradient Boosting)

In [ ]:
import xgboost as xgb

# Initialize XGBoost
xgb_model = xgb.XGBClassifier(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)

# Predictions and evaluation
y_pred_xgb = xgb_model.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

print("XGBoost Accuracy: ", accuracy_xgb)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))


XGBoost Accuracy:  0.7567567567567568
Confusion Matrix:
 [[ 33  32]
 [ 13 107]]
Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.51      0.59        65
           1       0.77      0.89      0.83       120

    accuracy                           0.76       185
   macro avg       0.74      0.70      0.71       185
weighted avg       0.75      0.76      0.74       185



This code initializes and trains an XGBoost classifier with 100 estimators. After training on the X_train and y_train data, it makes predictions on the test set (X_test). The accuracy, confusion matrix, and classification report for the predictions are then printed to evaluate the model's performance.

### Hyperparameter Tuning for Boosting

In [ ]:
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.01, 0.1]
}

grid_search_xgb = GridSearchCV(xgb.XGBClassifier(random_state=42), param_grid_xgb, cv=5)
grid_search_xgb.fit(X_train, y_train)

# Best parameters for XGBoost
print("Best Parameters (XGBoost): ", grid_search_xgb.best_params_)

Best Parameters (XGBoost):  {'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 100}


This code performs hyperparameter tuning for an XGBoost classifier using GridSearchCV. It searches for the best combination of n_estimators, max_depth, and learning_rate based on 5-fold cross-validation, and then prints the best parameters found.

## Conclusion

* This case study demonstrated the use of Decision Tree, Bagging (Random Forest), and Boosting (AdaBoost, Gradient Boosting, XGBoost) for a loan approval prediction task. These models are powerful techniques in supervised learning, each having its strengths in different scenarios. The best approach depends on the problem complexity, dataset size, and interpretability requirements.
   * Decision Trees are highly interpretable but suffer from overfitting, making them less suitable for complex real-world problems without ensemble methods.
   * Random Forest (Bagging) strikes a balance between bias and variance and is a good choice when a stable model with fewer parameters to tune is desired.
   * Boosting methods, particularly XGBoost, typically outperform other methods in terms of accuracy but come with increased computational cost and the need for more careful tuning.


### Final Recommendation

* Random Forest is recommended if you're looking for a robust and fairly quick-to-tune model for loan approval prediction.
* XGBoost should be your choice if accuracy is of the utmost importance and computational resources are available for fine-tuning the model.